In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(418)

In [6]:
df = df.drop_duplicates()
df.shape

(49582, 2)

# **Pre-processing**

* 1. converting to Lowercase

In [7]:
df["review"] = df["review"].str.lower()

* 2. Removing the URLs

In [8]:
import re # regular expression

def remove_url(text):
    re.sub(r"http\S+", "", text) # pattern, replace, string
    # \S -> non white space cheracter
    return text

df["review"] = df["review"].apply(remove_url)

* 3. Removing Punctuations

In [9]:
def remove_punc(text):
    re.sub(r"[^A-Za-z0-9\s]", "", text) # remove all outside from this (A-Z, a-z, 0-9, \s)    
    # \s -> white space cheracter
    return text

df["review"] = df["review"].apply(remove_punc)

* 4. Removing HTML tags

In [10]:
def remove_html(text):
    re.sub(r"<.*?>", "", text) # . -> mean any cheracter   * -> zero or more cheracter   ? -> mean non-greedy manner
    return text

df["review"] = df["review"].apply(remove_html)

* 5. Removing the Stopwords

In [11]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [13]:
# sample_text = "I like coding in python"
# tokens = word_tokenize(sample_text)
# tokens

In [14]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

* 6. Stemming

In [15]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)
        

* 7. Encoding

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [17]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod 'll hook . y rght...,1
1,wder ltle producti . < br / > < br / > film te...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli 's fmli lttle boy ( jke ) thk 's zomb c...,0
4,petter mttei 's `` love time mey '' vulli stun...,1


* 8. Vectorization

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000) # only top 5000 words

X = tf.fit_transform(df["review"])

In [19]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4118652 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3540)	0.05212272537350137
  (0, 2872)	0.08839013118552876
  (0, 4944)	0.11096996231835549
  (0, 3003)	0.5368331840636914
  (0, 1281)	0.13030066169544557
  (0, 2281)	0.09152218892711557
  (0, 1927)	0.07494991561573583
  (0, 3554)	0.09023767552333409
  (0, 1365)	0.058717555580018135
  (0, 1958)	0.058525907769782386
  (0, 220)	0.12250141880894057
  (0, 1620)	0.06936928233880027
  (0, 4378)	0.04095933187049547
  (0, 4179)	0.16858934880929974
  (0, 3697)	0.03182766309697489
  (0, 4747)	0.25253619116075937
  (0, 3807)	0.041664440077699906
  (0, 4780)	0.11082447019910512
  (0, 1739)	0.035214224368998244
  (0, 4504)	0.07232368729535452
  (0, 3860)	0.16646566618160646
  (0, 1629)	0.057670574139969275
  (0, 1859)	0.06798794620841928
  (0, 3332)	0.060908682996563546
  (0, 3335)	0.07947804655176903
  :	:
  (49581, 2765)	0.0858565979434153
  (49581, 4146)	0.0849794791

# **Dataset & DataLoaders**

* 1. TensorDataset

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, df["sentiment"], test_size = 0.2, random_state = 42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size = 0.3, random_state = 42
)

In [21]:
import torch
from torch.utils.data import DataLoader, TensorDataset

In [22]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2301540 stored elements and shape (27765, 5000)>

In [23]:
X_train = X_train.toarray() # sparse dtype to array
X_val = X_val.toarray()
X_test = X_test.toarray()

In [24]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(), # from_numpy -> convert numpy array to tensor
    torch.from_numpy(y_train.values).float()
)

val_set = TensorDataset(
    torch.from_numpy(X_val).float(),
    torch.from_numpy(y_val.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

* 2. DataLoader

In [25]:
train_loader = DataLoader(train_set, shuffle = True, batch_size = 64)
val_loader = DataLoader(val_set, shuffle = False, batch_size = 64)
test_loader = DataLoader(test_set, shuffle = False, batch_size = 64)

# **Build RNN**

In [26]:
import torch.nn as nn
import torch.optim as optim

In [27]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Rnn layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => (no of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.shape[0], self.hidden_size)

        output, _ = self.rnn(x, h0)
        # 1st hidden state for every timesteps => (batch size, seq_len, hidden size)
        # 2nd final hidden state of each RNN layer => (num_layers, batch_size, hidden_size)
        output = self.fc(output[:, -1, :])
        # (select all sample, select the last time steps, select all hidden features)
        return output

In [28]:
input_size = X_train.shape[1]

model = RNN(input_size)

In [29]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

# **Training RNN**

In [30]:
epochs = 100
patience = 5
epoch_without_improvement = 0
best_val_loss = float("inf")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1) # add singleton direction => 2D -> 3D
        outputs = model(xb).squeeze(-1) # plain simple output not in probability
        # (batch_size, 1), exxpect (batch_Size,), .squwwze(-1) fixed it by removes only the last dimension.

        loss = criterion(outputs, yb.float().view(-1)) # compute loss
        loss.backward() # backprop
        optimizer.step() # weight/parameter update

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # validation
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.unsqueeze(1)
            outputs = model(xb).squeeze(-1)
            loss = criterion(outputs, yb.float().view(-1))
            running_val_loss += loss.item()

    val_loss = running_val_loss / len(val_loader)

    print(f"epoch: {epoch + 1}/{epochs} ==> train loss = {train_loss}, val loss = {val_loss}")

    if val_loss < best_val_loss:
        epoch_without_improvement = 0
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pk")
        print("Validation loss improve. Save Model")

    else:
        epoch_without_improvement += 1
        print(f"No improvement: {epoch_without_improvement}/{patience}")


    if epoch_without_improvement >= patience:
        print("Early stop triggered")
        break    

epoch: 1/100 ==> train loss = 0.4124303598832425, val loss = 0.31287377543987765
Validation loss improve. Save Model
epoch: 2/100 ==> train loss = 0.261511835291089, val loss = 0.32147320151649494
No improvement: 1/5
epoch: 3/100 ==> train loss = 0.236119091751114, val loss = 0.33875831064357553
No improvement: 2/5
epoch: 4/100 ==> train loss = 0.22485992581956946, val loss = 0.3572033136602371
No improvement: 3/5
epoch: 5/100 ==> train loss = 0.21776214976739225, val loss = 0.37747511283684804
No improvement: 4/5
epoch: 6/100 ==> train loss = 0.2129409427707371, val loss = 0.3840188054788497
No improvement: 5/5
Early stop triggered


# **Evaluation**

In [31]:
model.load_state_dict(torch.load("best_model.pk"))

<All keys matched successfully>

In [32]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

y_true = []
y_pred = []

test_loss = 0.0

with torch.no_grad():

    for xb, yb in test_loader:

        xb = xb.unsqueeze(1)

        # Get raw logits
        outputs = model(xb).squeeze(-1)

        # Calculate test loss
        loss = criterion(outputs, yb.float().view(-1))
        test_loss += loss.item()

        # Convert logits into probabilities
        probabilities = torch.sigmoid(outputs)

        # Convert probabilities into binary predictions
        predictions = (probabilities >= 0.5).int()

        y_true.extend(yb.view(-1).cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

# Average test loss
test_loss /= len(test_loader)

# Evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))


Test Loss: 0.3103
Accuracy:  0.8670
Precision: 0.8552
Recall:    0.8849
F1 Score:  0.8698

Confusion Matrix:
[[4193  746]
 [ 573 4405]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.85      0.86      4939
         1.0       0.86      0.88      0.87      4978

    accuracy                           0.87      9917
   macro avg       0.87      0.87      0.87      9917
weighted avg       0.87      0.87      0.87      9917

